# OOP: Encapsulation

## 1. Definition
**Encapsulation** combines data (attributes) and the methods that operate on that data inside a class, while controlling how the object's internal state is accessed or changed.

Its purpose is to prevent accidental or invalid changes and keep an object in a valid state.

**Real-world example:** A bank account exposes `deposit()` and `withdraw()` rather than allowing callers to set its balance to any value.

## 2. Access conventions in Python

| Syntax | Meaning | Important detail |
|---|---|---|
| `name` | Public | Intended for normal external access |
| `_name` | Protected by convention | Signals internal or subclass use; not enforced |
| `__name` | Name-mangled | Helps avoid accidental access and naming conflicts; not truly private |

Python does not enforce strict private access. Double-leading underscores trigger name mangling, not security.

## 3. Public members
Public attributes and methods can be accessed directly. Use them when unrestricted access is safe.

In [ ]:
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price

    def show_details(self):
        print(f"{self.name}: ₹{self.price:.2f}")

product = Product("Keyboard", 1499)
product.show_details()

product.price = 1299
product.show_details()

## 4. Protected-by-convention members
A single leading underscore signals that an attribute or method is intended for internal use. Subclasses can still access it; Python does not block outside access.

In [ ]:
class Employee:
    def __init__(self, name, department):
        self.name = name
        self._department = department

class Developer(Employee):
    def show_department(self):
        print(f"{self.name} works in {self._department}")

developer = Developer("Aman", "Engineering")
developer.show_department()

## 5. Private members and name mangling
A double leading underscore causes name mangling. `__balance` inside `BankAccount` is internally renamed to a class-specific name such as `_BankAccount__balance`.

This reduces accidental access and naming conflicts. It is not a security mechanism.

In [ ]:
class BankAccount:
    def __init__(self, owner, opening_balance):
        self.owner = owner
        self.__balance = opening_balance

    def show_balance(self):
        print(f"{self.owner}: ₹{self.__balance:.2f}")

account = BankAccount("Riya", 5000)
account.show_balance()

# account.__balance raises AttributeError.
# Name-mangled access exists, but should not be used as the public interface.
# print(account._BankAccount__balance)

## 6. Getters, setters and @property
A getter reads a value; a setter validates and updates it. Python's `@property` provides attribute-style access while keeping control inside the class.

Use it when a value must follow a rule, such as a price that cannot be negative.

In [ ]:
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price  # Calls the setter.

    @property
    def price(self):
        return self.__price

    @price.setter
    def price(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("Price must be numeric.")
        if value < 0:
            raise ValueError("Price cannot be negative.")
        self.__price = float(value)

product = Product("Monitor", 12000)
print(product.price)

product.price = 11500
print(product.price)

# product.price = -500  # Raises ValueError

## 7. Encapsulation through business methods
Encapsulation is not just hiding attributes. Methods should enforce business rules whenever the object's state changes.

For a bank account, deposits and withdrawals are controlled operations. The balance is never assigned directly by outside code.

In [ ]:
class BankAccount:
    def __init__(self, holder, opening_balance=0):
        if opening_balance < 0:
            raise ValueError("Opening balance cannot be negative.")
        self.holder = holder
        self.__balance = float(opening_balance)

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Deposit must be positive.")
        self.__balance += amount
        return self.__balance

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("Withdrawal must be positive.")
        if amount > self.__balance:
            raise ValueError("Insufficient balance.")
        self.__balance -= amount
        return self.__balance

    @property
    def balance(self):
        return self.__balance

account = BankAccount("Aman", 10000)
account.deposit(2500)
account.withdraw(3000)
print(f"Balance: ₹{account.balance:.2f}")

## 8. Key benefits
- Protects object state from unintended changes.
- Allows validation at the point of modification.
- Hides implementation details behind a clear interface.
- Makes code easier to maintain and extend.
- Keeps business rules in one place.

## 9. Quick comparison

| Technique | Purpose |
|---|---|
| Public attribute | Direct access is acceptable |
| `_attribute` | Signals internal use by convention |
| `__attribute` | Name mangling to reduce accidental access |
| `@property` | Controlled attribute access |
| Public methods | Provide valid operations on internal state |

---

# Advanced Practice Problems

**Instructions:** Solve each problem independently. Use private attributes for internal state, expose only suitable methods/properties, validate inputs, and test both successful and failed operations. Each problem has its own Markdown cell followed immediately by a blank code cell.

## Problem 1 — Digital Banking and Transaction Management System

**Difficulty:** Advanced  
**Concepts:** Encapsulation, private attributes, properties, validation, transaction history, exception handling.

### Scenario
Build a banking module that supports deposits, withdrawals, transfers and transaction history. The balance must never be changed directly from outside the account.

### Requirements
1. Create `BankAccount` with `account_number`, `account_holder`, `__balance`, `__transaction_history` and `__is_active`.
2. Keep balance and transaction history private.
3. Implement `deposit(amount)`, `withdraw(amount)`, `transfer(target_account, amount)`, `get_balance()`, `get_transaction_history()` and `deactivate_account()`.
4. Amounts must be positive. A withdrawal cannot exceed the balance.
5. A transfer must debit the sender and credit the receiver. If either account is inactive or funds are insufficient, neither account's balance should change.
6. Record each successful operation with transaction type, amount, timestamp and description.
7. Return a copy or immutable view of transaction history; never return the internal list.
8. Use properties for suitable read-only account information.
9. Raise meaningful exceptions for invalid operations.
10. Demonstrate at least two accounts and successful and failed transactions.

### Test cases
- Open account A with ₹20,000 and account B with ₹5,000.
- Deposit ₹5,000 into A; withdraw ₹3,000.
- Transfer ₹4,000 from A to B.
- Attempt a withdrawal larger than the available balance.
- Deactivate B and attempt another transfer to it.
- Try modifying the returned transaction history and verify the internal history is unchanged.

**Expected outcome:** Balances remain valid, failed transfers do not partially execute, and successful operations are recorded.

## Problem 2 — E-commerce Inventory and Shopping Cart System

**Difficulty:** Advanced  
**Concepts:** Encapsulation, private attributes, properties, validation, object interaction, inventory consistency.

### Scenario
Build an online store where product prices, stock and cart contents cannot be changed arbitrarily. All changes must pass through controlled operations.

### Requirements
1. Create `Product` with `product_id`, `name`, `__price` and `__stock_quantity`.
2. Expose `price` through a validated property. Make stock read-only externally.
3. Implement `update_price()`, `add_stock()`, `reduce_stock()` and `get_product_details()`.
4. Create `ShoppingCart` with private `__cart_items` and `__discount_percentage`.
5. Implement `add_product()`, `remove_product()`, `update_quantity()`, `apply_discount()`, `calculate_total()` and `generate_bill()`.
6. Price must be positive; stock cannot become negative; quantity must be a positive integer; discount must be between 0 and 50%.
7. Prevent callers from modifying the cart's internal collection.
8. Define a consistent stock policy: reserve stock when an item is added; restore it when removed; finalize the reservation on checkout. Avoid double deduction.
9. Generate an itemized bill with name, quantity, unit price, subtotal, discount and final total.
10. Test multiple products, invalid quantities, insufficient stock, removal and checkout.

### Test cases
- Laptop: ₹55,000, stock 10.
- Mouse: ₹800, stock 20.
- Add 2 laptops and 3 mice.
- Apply a 10% discount and update the mouse quantity.
- Attempt to add more laptops than available.
- Remove an item, then checkout and verify stock and bill totals.

**Expected outcome:** Product and cart state can only change through validated operations, and inventory remains consistent.

## Problem 3 — Employee Compensation and Payroll Management

**Difficulty:** Advanced  
**Concepts:** Encapsulation, private attributes, read-only properties, validation, controlled state changes, class interaction.

### Scenario
Create a payroll system in which salary, performance rating, bonus and employment status are protected. HR operations must update these values through approved methods.

### Requirements
1. Create `Employee` with `employee_id`, `name`, `__monthly_salary`, `__performance_rating`, `__bonus` and `__employment_status`.
2. Keep salary, rating, bonus and status private.
3. Provide read-only properties for employee ID, name, salary, rating and status.
4. Implement `update_salary()`, `update_performance_rating()`, `calculate_bonus()`, `calculate_annual_salary()` and `change_employment_status()`.
5. Bonus is based on annual base salary:
   - Rating 5: 20%
   - Rating 4: 15%
   - Rating 3: 10%
   - Rating 2: 5%
   - Rating 1: 0%
6. Salary must be positive; rating must be an integer from 1 to 5; status must be one of the allowed values.
7. An inactive employee cannot receive a salary update.
8. Create `PayrollSystem` to add and find employees, calculate total payroll and generate a report.
9. Do not expose the internal employee collection.
10. Test salary changes, rating changes, bonus calculations, invalid values and inactive employees.

### Test cases
- Employee 101: ₹40,000 monthly salary, rating 5.
- Employee 102: ₹60,000 monthly salary, rating 3.
- Employee 103: ₹35,000 monthly salary, rating 2.
- Calculate annual salary, bonus and total compensation.
- Attempt rating 6 and a negative salary.
- Deactivate an employee and attempt to update salary.

**Expected outcome:** Payroll calculations remain correct and employee state changes only through validated operations.

## Problem 4 — Hospital Patient Records and Appointment Management

**Difficulty:** Advanced  
**Concepts:** Encapsulation, private data, controlled access, validation, composition, defensive copying.

### Scenario
Build a hospital system that manages patient profiles, medical records and appointments. Internal records must not be exposed for arbitrary modification.

### Requirements
1. Create `Patient` with `patient_id`, `name`, `__phone_number`, `__medical_history` and `__appointments`.
2. Keep phone number, medical history and appointments private.
3. Implement `update_phone_number()`, `add_medical_record()`, `get_medical_history()`, `book_appointment()`, `cancel_appointment()` and `get_appointments()`.
4. Create an `Appointment` class with private appointment ID, doctor, date/time and status.
5. Allow only these appointment states: Scheduled, Completed and Cancelled.
6. A cancelled appointment cannot be completed; a completed appointment cannot be cancelled.
7. Validate phone number format and reject appointments in the past.
8. Prevent duplicate active appointments for the same doctor and time.
9. Return copies or immutable views of medical history and appointment collections.
10. Create `Hospital` to register and find patients without exposing its internal patient collection.

### Test cases
- Register two patients and add medical records.
- Book appointments with different doctors.
- Attempt to book the same doctor at the same time twice.
- Cancel a scheduled appointment and try to complete it.
- Try modifying a returned medical-history collection and verify the original is unchanged.

**Expected outcome:** Patient records and appointment states change only through valid operations.

**Privacy note:** Use fictional patient data for this exercise.

## Problem 5 — Secure User Authentication and Account Management

**Difficulty:** Advanced  
**Concepts:** Encapsulation, private attributes, properties, validation, state management, exception handling, security-aware design.

### Scenario
Build a basic authentication module. Password data, failed login attempts and account-lock status must be protected and changed only through controlled operations.

### Requirements
1. Create `UserAccount` with `username`, `__password_hash`, `__failed_login_attempts`, `__is_locked` and `__last_login`.
2. Never store or expose plain-text passwords.
3. Use a suitable password-hashing library with a password-specific algorithm such as Argon2id, scrypt or bcrypt. Do not use plain MD5 or SHA-1.
4. Implement `set_password()`, `verify_password()`, `login()`, `logout()`, `reset_failed_attempts()` and `is_account_locked()`.
5. Enforce a documented password policy.
6. Lock the account after 5 consecutive failed attempts.
7. A successful login resets the failed-attempt counter and updates the last-login time.
8. A locked account cannot log in until an authorized reset operation unlocks it.
9. Expose only safe read-only information through properties. Never expose the password hash.
10. Create `AuthenticationSystem` to register users, authenticate login requests and perform authorized resets. Do not expose its internal user collection.
11. Return safe error messages that do not reveal whether a username exists.
12. Test registration, successful login, failed login, account locking, password change and authorized unlocking.

### Test cases
- Register `gaurav_dev` and set a valid password.
- Log in with the correct password.
- Make 5 consecutive incorrect login attempts.
- Verify that the account is locked, even if the correct password is then supplied.
- Perform an authorized reset and log in again.
- Verify that no public interface returns the password or password hash.

**Expected outcome:** Authentication state remains encapsulated and all state transitions follow the defined security rules.

**Security note:** This is an OOP exercise, not production-ready authentication. Real systems also require secure recovery, rate limiting, secure session management and established authentication frameworks.